# Price-149 external benchmark

Historical benchmark workflow adapted to portable paths.
Inference, filtering and retraining are disabled by default.
The original alignment file is required for the optional filtering check.
Reported historical results are documented in results/price149/.
This notebook is not a verified reproduction of the external benchmark.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pca_transform.py").is_file()
)
sys.path.insert(0, str(ROOT))

DATA = Path(os.environ.get(
    "PBL_DATA_ROOT", ROOT.parent / "enzyme-classification-data"
))
PRICE = DATA / "price_149"
SPLIT = DATA / "task_2/split"
OUTPUT = ROOT / "outputs/price149"

PREFIX = "exp_enzymes_c30_flens2-98_usamp2000_exp"

TRAIN_FASTA = SPLIT / f"{PREFIX}_train85.fasta"
FILTERED_FASTA = SPLIT / f"{PREFIX}_train85_filtered_price.fasta"

MATCHES = Path(os.environ.get(
    "PBL_PRICE_MATCHES", PRICE / "matches_2.m8"
))

RUN_INFERENCE = False
RUN_FILTER_CHECK = False
RUN_TRAINING = False


In [ ]:
import pandas as pd
from Bio import SeqIO
from task_2.src.preprocessing import (
    create_dataset_from_h5,
    create_dataset_from_fasta,
    filter_h5_from_fasta,
    dim_reduction,
)
from task_2.src.infer import infer, infer_fasta
from task_2.src.utils import df_to_fasta


In [ ]:
records = SeqIO.parse(TRAIN_FASTA, format="fasta")

In [ ]:
train_set_seq_list = list()
for rec in records:
    train_set_seq_list.append(str(rec.seq))

In [ ]:
price_df = pd.read_csv(PRICE / "price.csv", sep="\t")

In [ ]:
price_df['EC number'] = price_df['EC number'].apply(lambda x : int(x.split('.')[0]))
price_df = price_df.rename(columns={'EC number': 'EC main class'})
price_df

In [ ]:
if RUN_INFERENCE:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    df_to_fasta(price_df, path=OUTPUT / "price_test_set.fasta")



In [ ]:
if RUN_INFERENCE:
    # Original inference attempt with the regular Task 2 model.
    infer_fasta(
        "task2",
        OUTPUT / "price_test_set.fasta",
        metrics=True,
        known_seqs=False,
    )



In [ ]:
if RUN_FILTER_CHECK:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    from Bio import SeqIO

    # 1. Load the IDs of sequences to ban
    banned_ids = set()

    # Open the BLAST/MMseqs output file
    # Format is: PriceID  TrainID  Identity
    with open(MATCHES, "r") as f:
        for line in f:
            parts = line.strip().split()
            train_id = parts[1] # The ID from your training set
            identity = float(parts[2]) # The % identity
        
            # Double check the threshold (optional, as tool did it)
            if identity >= 0.30: # 30%
                banned_ids.add(train_id)

    print(f"Found {len(banned_ids)} contaminated sequences to remove.")

    # 2. Filter the original FASTA file
    original_fasta = FILTERED_FASTA
    clean_fasta = OUTPUT / "training_set_CLEANED.fasta"

    count_kept = 0
    count_removed = 0

    with open(clean_fasta, "w") as out_handle:
        for record in SeqIO.parse(original_fasta, "fasta"):
            if record.id in banned_ids:
                count_removed += 1
            else:
                SeqIO.write(record, out_handle, "fasta")
                count_kept += 1

    print(f"Done! Kept {count_kept} sequences. Removed {count_removed}.")


In [ ]:
if RUN_FILTER_CHECK:
    records = SeqIO.parse(FILTERED_FASTA, format="fasta")


In [ ]:
if RUN_FILTER_CHECK:
    with open(OUTPUT / "filtered_price_checked.fasta", "w") as out_fasta:
        for rec in records:
            if rec.id.split("|")[1] not in banned_ids:
                SeqIO.write(rec, out_fasta, format="fasta")



In [ ]:
if RUN_TRAINING:
    OUTPUT.mkdir(parents=True, exist_ok=True)

    folder = DATA / "task_2" / f"embeddings_uniprot_{PREFIX}"
    raw_train = folder / f"{folder.name}_train85.h5"
    filtered_h5 = OUTPUT / "filtered_training_embeddings.h5"

    if filtered_h5.exists():
        raise FileExistsError(filtered_h5)

    filter_h5_from_fasta(
        raw_train,
        FILTERED_FASTA,
        out_path=filtered_h5,
    )

    train_set_filtered = create_dataset_from_h5(
        filtered_h5, unsqueeze=False
    )
    assert len(train_set_filtered) == 6581



In [ ]:
if RUN_TRAINING:
    dim_reduction(
        train_set_filtered,
        128,
        ROOT / "checkpoints/pca/task2_pca_original.npz",
    )



In [ ]:
if RUN_TRAINING:
    from task_2.src.models import Task2Model
    import torch.optim as optim
    from task_2.src.utils import get_class_distribution_list
    import torch
    import torch.nn as nn
    from task_2.src.train import train_model

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = Task2Model(dims=[128, 256, 7], dropouts=[0.11764897516223502], activation=nn.LeakyReLU())
    lr = 0.010407922166686873
    weight_decay = 0.03424566834898443
    label_smoothing = 0.066
    weights_alpha = 0.496
    weight_power = 0.483
    optimizer = optim.AdamW(params=model.parameters(), weight_decay=weight_decay)

    class_dist_list = get_class_distribution_list(train_set_filtered, relative=False)
    counts = torch.tensor(class_dist_list, dtype=torch.float32)

    w = (counts.sum() / (7.0 * torch.clamp(counts, min=1.0))).pow(weight_power)
    w = (w / w.mean()).to(device)

    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=label_smoothing)

    train_model(model=model, model_name=f'task_2_first_model_{train_set_filtered.name}', dataset=train_set_filtered, optimizer=optimizer, criterion=criterion, epochs=200, early_stopping=True, patience=12, batch_size=16, sampler_alpha=weights_alpha)
